# Configuraciones de los 600 modelos más descargados del Hub

Cuaderno de lectura de la medición `transformers/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Respalda el artículo [antes-de-la-primera-capa](https://manpla.net/posts/antes-de-la-primera-capa/). Carga el fichero de al lado —o lo descarga del repositorio si se ejecuta fuera de él—, muestra la ficha de procedencia y dibuja una figura con matplotlib a secas. Solo lee; no vuelve a tomar la instantánea: para eso está `generar.py`.

*Reading notebook for this measurement: loads the file next to it, prints the provenance record and draws one figure. Column names are in Spanish; `GLOSARIO.md` gives the English form.*

In [ ]:
import io, json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/transformers/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")


## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## El dato

In [ ]:
d = leer("configs.csv")
ok = d.dropna(subset=["params_reales", "params_predichos"]); ok = ok[ok.params_reales > 1e6]
print(len(d), "modelos ·", len(ok), "con recuento real y predicho")
d.clase.value_counts()

## Una figura

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(ok.params_reales, ok.params_predichos, s=12, alpha=.6)
lim = [ok.params_reales.min(), ok.params_reales.max()]
ax.plot(lim, lim, color="C3", lw=1, label="predicho = real")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("parámetros declarados"); ax.set_ylabel("parámetros predichos desde config.json"); ax.legend(); ax.set_title("la fórmula frente al recuento real")